In [1]:
from shearnet.core.dataset import generate_dataset, split_combined_images
from shearnet.utils.metrics import eval_ngmix, fork_eval_model

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from typing import Optional
from shearnet.cli.evaluate import load_config, create_parser, initialize_model

In [2]:
"""
Generate the dataset
"""

test_target_images, test_target_labels, obs = generate_dataset(
        samples=5000,
        psf_sigma=0.25,
        type='gauss',
        exp='ideal',
        seed=42,
        nse_sd=12.719674,
        npix=53,
        scale=0.141,
        return_psf=True,
        return_clean=False,
        return_obs=True,
    )

test_galaxy_images, test_psf_images = split_combined_images(test_target_images, has_psf=True, has_clean=False)

100%|██████████| 5000/5000 [00:12<00:00, 395.95it/s]


In [3]:
"""
Analyze the predicted flux of ngmix
"""

ngmix_metrics = eval_ngmix(test_obs=obs, test_labels=test_target_labels, seed=42, psf_model='gauss', gal_model='gauss')

for i in range(0,10,1):
    print(test_target_labels[i])
print('VS')
for i in range(0,10,1):
    print(ngmix_metrics['preds'][i])

Starting NGmix ML fitting: num_gal: 5000 | psf_model: gauss | gal_model: gauss | num_cores: 96

=== Combined Metrics (NGmix) ===
Mean Squared Error (MSE) from NGmix: 1.332017e+04
Average Bias from NGmix: 1.857997e+00
Time taken: 105.95 seconds

=== Per-Label Metrics ===
             g1: MSE = 1.554866e-04, Bias = +2.755719e-03
             g2: MSE = 1.517922e-04, Bias = +1.820223e-03
  g1g2_combined: MSE = 1.536394e-04, Bias = +2.287971e-03
          sigma: MSE = 1.015027e-02, Bias = +8.667410e-02
           flux: MSE = 5.328068e+04, Bias = +7.340738e+00

[ 7.9226442e-02 -2.7039587e-01  5.0000000e-01  1.2258970e+04]
[1.9511731e-01 2.4454683e-01 5.0000000e-01 1.2258970e+04]
[-5.0726914e-01 -3.3856666e-01  5.0000000e-01  1.2258970e+04]
[ 3.3238504e-02 -8.2223073e-02  5.0000000e-01  1.2258970e+04]
[-4.3683010e-03 -2.2179142e-01  5.0000000e-01  1.2258970e+04]
[2.2864348e-01 2.0222591e-01 5.0000000e-01 1.2258970e+04]
[1.7167982e-02 2.9308271e-01 5.0000000e-01 1.2258970e+04]
[ 1.2155243e-01 

In [4]:
"""
Trying ShearNet
"""

parser = create_parser()
args = parser.parse_args(["--model_name", "first_validation_test", "--test_samples", "5000"])
config = load_config(args)

state = initialize_model(config, test_galaxy_images, test_psf_images)

nn_metrics = fork_eval_model(state=state, test_images=test_galaxy_images, test_psf_images=test_psf_images, test_labels=test_target_labels)

for i in range(0,10,1):
    print(test_target_labels[i])
print('VS')
for i in range(0,10,1):
    print(nn_metrics['all_preds'][i])


Loading model config from: /home/adfield/ShearNet/plots/first_validation_test/training_config.yaml

Evaluation Configuration

evaluation:
  test_samples: 30000
  seed: 58

model:
  process_psf: True
  type: fork-like
  galaxy: {'type': 'research_backed'}
  psf: {'type': 'forklens_psf'}

plotting:
  plot: True

comparison:
  mcal: True
  ngmix: True
  psf_model: gauss
  gal_model: gauss


Loading Model
Found 1 matching checkpoint(s):
  1. first_validation_test162

Loading checkpoint from: /home/adfield/ShearNet/model_checkpoint/first_validation_test162
✓ Model checkpoint loaded successfully

=== Combined Metrics (ShearNet) ===
Mean Squared Error (MSE) from ShearNet: 7.595505e-01
Average Bias from ShearNet: -3.454612e-01
Time taken: 5.93 seconds

=== Per-Label Metrics ===
             g1: MSE = 2.095513e-01, Bias = +6.195257e-01
             g2: MSE = 1.211644e-01, Bias = +4.505321e-01
  g1g2_combined: MSE = 1.653579e-01, Bias = +5.350289e-01
          sigma: MSE = 2.252924e-02, Bias = 